In [1]:
import kagglehub
import shutil
from pathlib import Path
import pandas as pd
import psycopg

# Datenaufbereitung

Zunächst werden die einzelnen Dateien von Kaggle in einen lokalen Ordner heruntergeladen, entsprechend verschoben und anschließend die CSV-Dateien in DataFrames eingelesen.

In [2]:
try:
    df_circuits                 = pd.read_csv('./csv-data/circuits.csv')
    df_constructor_results      = pd.read_csv('./csv-data/constructor_results.csv')
    df_constructor_standings    = pd.read_csv('./csv-data/constructor_standings.csv')
    df_constructors             = pd.read_csv('./csv-data/constructors.csv')
    df_driver_standings         = pd.read_csv('./csv-data/driver_standings.csv')
    df_drivers                  = pd.read_csv('./csv-data/drivers.csv')
    df_lap_times                = pd.read_csv('./csv-data/lap_times.csv')
    df_pit_stops                = pd.read_csv('./csv-data/pit_stops.csv')
    df_qualifying               = pd.read_csv('./csv-data/qualifying.csv')
    df_races                    = pd.read_csv('./csv-data/races.csv')
    df_results                  = pd.read_csv('./csv-data/results.csv')
    df_seasons                  = pd.read_csv('./csv-data/seasons.csv')
    df_sprint_results           = pd.read_csv('./csv-data/sprint_results.csv')
    df_status                   = pd.read_csv('./csv-data/status.csv')

except:
    download_path = kagglehub.dataset_download("rohanrao/formula-1-world-championship-1950-2020")

    download_dir = Path(download_path)
    target_dir = Path("./csv-data")
    target_dir.mkdir(exist_ok=True)

    for file in download_dir.rglob("*.csv"):
        shutil.copy(file, target_dir / file.name)

    df_circuits                 = pd.read_csv('./csv-data/circuits.csv')
    df_constructor_results      = pd.read_csv('./csv-data/constructor_results.csv')
    df_constructor_standings    = pd.read_csv('./csv-data/constructor_standings.csv')
    df_constructors             = pd.read_csv('./csv-data/constructors.csv')
    df_driver_standings         = pd.read_csv('./csv-data/driver_standings.csv')
    df_drivers                  = pd.read_csv('./csv-data/drivers.csv')
    df_lap_times                = pd.read_csv('./csv-data/lap_times.csv')
    df_pit_stops                = pd.read_csv('./csv-data/pit_stops.csv')
    df_qualifying               = pd.read_csv('./csv-data/qualifying.csv')
    df_races                    = pd.read_csv('./csv-data/races.csv')
    df_results                  = pd.read_csv('./csv-data/results.csv')
    df_seasons                  = pd.read_csv('./csv-data/seasons.csv')
    df_sprint_results           = pd.read_csv('./csv-data/sprint_results.csv')
    df_status                   = pd.read_csv('./csv-data/status.csv')

#Verbindung zur Datenbank erstellen

In [3]:
connection = psycopg.connect(
    dbname='defaultdb',
    user='root',
    host='cockroach',
    port='26257',
    sslmode='disable'
)

cursor = connection.cursor()

# Tabellen erstellen

In [4]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS circuits (
    id          bigint          NOT NULL PRIMARY KEY DEFAULT unique_rowid(),
    circuit_ref varchar(255)    NOT NULL,
    name        varchar(255)    NOT NULL,
    location    varchar(255)    NOT NULL,
    country     varchar(255)    NOT NULL,
    lat         numeric         NOT NULL,
    lng         numeric         NOT NULL,
    alt         bigint          NOT NULL,
    url         varchar(255)    NOT NULL
);
""")

## Indexes
cursor.execute("""
    CREATE INDEX idx_circuit_ref_on_circuits
        ON circuits (circuit_ref);
""")

## Unique constraints
cursor.execute("""
    ALTER TABLE IF EXISTS circuits
        ADD CONSTRAINT circuit_ref_unique_on_circuits UNIQUE (circuit_ref);
""")

cursor.execute("""
    ALTER TABLE IF EXISTS circuits
        ADD CONSTRAINT name_unique_on_circuits UNIQUE (name);
""")

##########################################################################################

cursor.execute("""
CREATE TABLE IF NOT EXISTS races (
    id          bigint          NOT NULL PRIMARY KEY DEFAULT unique_rowid(),
    year        bigint          NOT NULL,
    round       bigint          NOT NULL,
    circuit_id  bigint          NOT NULL,
    name        varchar(255)    NOT NULL,
    date        date            NOT NULL,
    time        time,
    url         varchar(255),
    fp1_date    date,
    fp1_time    time,
    fp2_date    date,
    fp2_time    time,
    fp3_date    date,
    fp3_time    time,
    quali_date  date,
    quali_time  time,
    sprint_date date,
    sprint_time time,
    FOREIGN KEY (circuit_id) REFERENCES circuits (id) ON DELETE CASCADE
);
""")

## Indexes
cursor.execute("""
    CREATE INDEX idx_name_on_races
        ON races (name);
""")

## Unique constraints

##########################################################################################

cursor.execute("""
CREATE TABLE IF NOT EXISTS constructors (
    id              bigint          NOT NULL PRIMARY KEY DEFAULT unique_rowid(),
    constructor_ref varchar(255)    NOT NULL,
    name            varchar(255)    NOT NULL,
    nationality     varchar(255)    NOT NULL,
    url             varchar(255)    NOT NULL
);
""")

## Indexes
cursor.execute("""
    CREATE INDEX idx_constructor_ref_on_constr
        ON constructors (constructor_ref);
""")

## Unique constraints
cursor.execute("""
    ALTER TABLE IF EXISTS constructors
        ADD CONSTRAINT constr_ref_unique_on_constr UNIQUE (constructor_ref);
""")

cursor.execute("""
    ALTER TABLE IF EXISTS constructors
        ADD CONSTRAINT name_unique_on_constr UNIQUE (name);
""")

##########################################################################################

cursor.execute("""
CREATE TABLE IF NOT EXISTS drivers (
    id              bigint          NOT NULL PRIMARY KEY DEFAULT unique_rowid(),
    driver_ref      varchar(255)    NOT NULL,
    number          bigint,
    code            varchar(255),
    forename        varchar(255)    NOT NULL,
    surname         varchar(255)    NOT NULL,
    date_of_birth   date            NOT NULL,
    nationality     varchar(255)    NOT NULL,
    url             varchar(255)    NOT NULL
);
""")

## Indexes
cursor.execute("""
    CREATE INDEX idx_driver_ref_on_drivers
        ON drivers (driver_ref);
""")

## Unique constraints
cursor.execute("""
    ALTER TABLE IF EXISTS drivers
        ADD CONSTRAINT driver_ref_unique_on_drivers UNIQUE (driver_ref);
""")

##########################################################################################

cursor.execute("""
CREATE TABLE IF NOT EXISTS seasons (
    id      bigint          NOT NULL PRIMARY KEY DEFAULT unique_rowid(),
    year    bigint          NOT NULL,
    url     varchar(255)    NOT NULL
);
""")

## Indexes

## Unique constraints

##########################################################################################

cursor.execute("""
CREATE TABLE IF NOT EXISTS status (
    id      bigint          NOT NULL PRIMARY KEY DEFAULT unique_rowid(),
    status  varchar(255)    NOT NULL
);
""")

## Indexes
cursor.execute("""
    CREATE INDEX idx_status_on_status
        ON status (status);
""")

## Unique constraints
cursor.execute("""
    ALTER TABLE IF EXISTS status
        ADD CONSTRAINT status_unique_on_status UNIQUE (status);
""")

##########################################################################################

cursor.execute("""
CREATE TABLE IF NOT EXISTS qualifyings (
    id              bigint      NOT NULL PRIMARY KEY DEFAULT unique_rowid(),
    race_id         bigint      NOT NULL,
    driver_id       bigint      NOT NULL,
    constructor_id  bigint      NOT NULL,
    number          bigint      NOT NULL,
    position        bigint      NOT NULL,
    q1              interval,
    q2              interval,
    q3              interval,
    FOREIGN KEY (race_id) REFERENCES races (id) ON DELETE CASCADE,
    FOREIGN KEY (driver_id) REFERENCES drivers (id) ON DELETE CASCADE,
    FOREIGN KEY (constructor_id) REFERENCES constructors (id) ON DELETE CASCADE
);
""")

## Indexes

## Unique constraints

##########################################################################################

cursor.execute("""
CREATE TABLE IF NOT EXISTS pit_stops (
    id              bigint      NOT NULL PRIMARY KEY DEFAULT unique_rowid(),
    race_id         bigint      NOT NULL,
    driver_id       bigint      NOT NULL,
    stop            bigint      NOT NULL,
    lap             bigint      NOT NULL,
    time            time        NOT NULL,
    duration        interval    NOT NULL,
    milliseconds    bigint      NOT NULL,
    FOREIGN KEY (race_id) REFERENCES races (id) ON DELETE CASCADE,
    FOREIGN KEY (driver_id) REFERENCES drivers (id) ON DELETE CASCADE
);
""")

## Indexes

## Unique constraints

##########################################################################################

cursor.execute("""
CREATE TABLE IF NOT EXISTS results (
    id                  bigint          NOT NULL PRIMARY KEY DEFAULT unique_rowid(),
    race_id             bigint          NOT NULL,
    driver_id           bigint          NOT NULL,
    constructor_id      bigint          NOT NULL,
    number              bigint,
    grid                bigint          NOT NULL,
    position            bigint,
    position_text       varchar(255)    NOT NULL,
    position_order      bigint          NOT NULL,
    points              numeric         NOT NULL,
    laps                bigint          NOT NULL,
    time                interval,
    milliseconds        bigint,
    fastest_lap         bigint,
    rank                bigint,
    fastest_lap_time    interval,
    fastest_lap_speed   numeric,
    status_id           bigint          NOT NULL,
    FOREIGN KEY (race_id) REFERENCES races (id) ON DELETE CASCADE,
    FOREIGN KEY (driver_id) REFERENCES drivers (id) ON DELETE CASCADE,
    FOREIGN KEY (status_id) REFERENCES status (id) ON DELETE CASCADE
);
""")

## Indexes

## Unique constraints

##########################################################################################

cursor.execute("""
CREATE TABLE IF NOT EXISTS lap_times (
    id              bigint      NOT NULL PRIMARY KEY DEFAULT unique_rowid(),
    race_id         bigint      NOT NULL,
    driver_id       bigint      NOT NULL,
    lap             bigint      NOT NULL,
    position        bigint      NOT NULL,
    time            interval    NOT NULL,
    milliseconds    bigint      NOT NULL,
    FOREIGN KEY (race_id) REFERENCES races (id) ON DELETE CASCADE,
    FOREIGN KEY (driver_id) REFERENCES drivers (id) ON DELETE CASCADE
);
""")

## Indexes

## Unique constraints

##########################################################################################

cursor.execute("""
CREATE TABLE IF NOT EXISTS driver_standings (
    id              bigint          NOT NULL PRIMARY KEY DEFAULT unique_rowid(),
    race_id         bigint          NOT NULL,
    driver_id       bigint          NOT NULL,
    points          bigint          NOT NULL,
    position        bigint          NOT NULL,
    position_text   varchar(255)    NOT NULL,
    wins            bigint          NOT NULL,
    FOREIGN KEY (race_id) REFERENCES races (id) ON DELETE CASCADE,
    FOREIGN KEY (driver_id) REFERENCES drivers (id) ON DELETE CASCADE
);
""")

## Indexes

## Unique constraints

##########################################################################################

cursor.execute("""
CREATE TABLE IF NOT EXISTS sprint_results (
    id              bigint          NOT NULL PRIMARY KEY DEFAULT unique_rowid(),
    result_id       bigint          NOT NULL,
    race_id         bigint          NOT NULL,
    driver_id       bigint          NOT NULL,
    constructor_id  bigint          NOT NULL,
    number          bigint          NOT NULL,
    grid            bigint          NOT NULL,
    position        bigint,
    FOREIGN KEY (result_id) REFERENCES results (id) ON DELETE CASCADE,
    FOREIGN KEY (constructor_id) REFERENCES constructors (id) ON DELETE CASCADE,
    FOREIGN KEY (race_id) REFERENCES races (id) ON DELETE CASCADE,
    FOREIGN KEY (driver_id) REFERENCES drivers (id) ON DELETE CASCADE
);
""")

## Indexes

## Unique constraints

##########################################################################################

cursor.execute("""
CREATE TABLE IF NOT EXISTS constructor_standings (
    id              bigint          NOT NULL PRIMARY KEY DEFAULT unique_rowid(),
    race_id         bigint          NOT NULL,
    constructor_id  bigint          NOT NULL,
    points          bigint          NOT NULL,
    position        bigint          NOT NULL,
    position_text   varchar(255)    NOT NULL,
    wins            bigint          NOT NULL,
    FOREIGN KEY (race_id) REFERENCES races (id) ON DELETE CASCADE,
    FOREIGN KEY (constructor_id) REFERENCES constructors (id) ON DELETE CASCADE
);
""")

## Indexes

## Unique constraints

##########################################################################################

cursor.execute("""
CREATE TABLE IF NOT EXISTS constructor_results (
    id              bigint          NOT NULL PRIMARY KEY DEFAULT unique_rowid(),
    race_id         bigint          NOT NULL,
    constructor_id  bigint          NOT NULL,
    points          bigint          NOT NULL,
    status          varchar(255),
    FOREIGN KEY (constructor_id) REFERENCES constructors (id) ON DELETE CASCADE,
    FOREIGN KEY (race_id) REFERENCES races (id) ON DELETE CASCADE
);
""")

## Indexes

## Unique constraints

<psycopg.Cursor [COMMAND_OK] [IDLE] (host=localhost port=26257 user=root database=defaultdb) at 0x264f794e2a0>

# Hilfsmethoden
Die Funktion `safe(val)` dient dazu, spezielle Platzhalter für fehlende Werte wie \N, <defer> oder auch NaN in None zu konvertieren. Dadurch wird sichergestellt, dass beim späteren Einfügen in die Datenbank korrekt ein `NULL`-Wert gesetzt wird, anstatt ungültiger Zeichenketten oder undefinierter Werte.

In [5]:
def safe(val):
    if pd.isna(val):
        return None
    if isinstance(val, str) and val in ["\\N", "<defer>"]:
        return None
    return val

# Tabellen befüllen
Die aus den CSV-Dateien geladenen Daten werden zeilenweise aus den entsprechenden DataFrames extrahiert und in vorbereiteten Listen gespeichert. Um die Daten effizient in die zugehörigen Datenbanktabellen zu laden, wird die Methode executemany verwendet. Diese ermöglicht das Ausführen mehrerer INSERT-Anweisungen in einem Schritt und spart dadurch im Vergleich zu Einzeloperationen erheblich Zeit.

Dabei wird auf jede Zelle der Daten die zuvor definierte safe-Funktion angewendet. Diese stellt sicher, dass fehlende oder ungültige Werte korrekt als NULL in der Datenbank gespeichert werden.

In [6]:
circuit_values = [
    (
        safe(row['circuitId']),
        safe(row['circuitRef']),
        safe(row['name']),
        safe(row['location']),
        safe(row['country']),
        safe(row['lat']),
        safe(row['lng']),
        safe(row['alt']),
        safe(row['url'])
    )
    for _, row in df_circuits.iterrows()
]

cursor.executemany("""
    INSERT INTO circuits (id, circuit_ref, name, location, country, lat, lng, alt, url)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
""", circuit_values)

In [7]:
race_values = [
    (
        safe(row['raceId']),
        safe(row['year']),
        safe(row['round']),
        safe(row['circuitId']),
        safe(row['name']),
        safe(row['date']),
        safe(row['time']),
        safe(row['url']),
        safe(row['fp1_date']),
        safe(row['fp1_time']),
        safe(row['fp2_date']),
        safe(row['fp2_time']),
        safe(row['fp3_date']),
        safe(row['fp3_time']),
        safe(row['quali_date']),
        safe(row['quali_time']),
        safe(row['sprint_date']),
        safe(row['sprint_time'])
    )
    for _, row in df_races.iterrows()
]

cursor.executemany(
    """
        INSERT INTO races (
            id, year, round, circuit_id, name, date, time, url,
            fp1_date, fp1_time, fp2_date, fp2_time, fp3_date, fp3_time,
            quali_date, quali_time, sprint_date, sprint_time
        ) VALUES (
            %s, %s, %s, %s, %s, %s, %s, %s,
            %s, %s, %s, %s, %s, %s,
            %s, %s, %s, %s
        )
    """, race_values
)

In [8]:
# Create a list of tuples with the constructor data
constructor_values = [
    (
        safe(row['constructorId']),
        safe(row['constructorRef']),
        safe(row['name']),
        safe(row['nationality']),
        safe(row['url'])
    )
    for _, row in df_constructors.iterrows()
]

# Bulk insert with executemany
cursor.executemany("""
    INSERT INTO constructors (id, constructor_ref, name, nationality, url)
    VALUES (%s, %s, %s, %s, %s)
""", constructor_values)


In [9]:
driver_values = [
    (
        safe(row['driverId']),
        safe(row['driverRef']),
        safe(row['number']),
        safe(row['code']),
        safe(row['forename']),
        safe(row['surname']),
        safe(row['dob']),
        safe(row['nationality']),
        safe(row['url'])
    )
    for _, row in df_drivers.iterrows()
]

cursor.executemany("""
    INSERT INTO drivers (id, driver_ref, number, code, forename, surname, date_of_birth, nationality, url)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
""", driver_values)


In [10]:
season_values = [
    (
        safe(row['year']),
        safe(row['url'])
    )
    for _, row in df_seasons.iterrows()
]

cursor.executemany("""
    INSERT INTO seasons (year, url)
    VALUES (%s, %s)
""", season_values)


In [11]:
qualifying_values = [
    (
        safe(row['qualifyId']),
        safe(row['raceId']),
        safe(row['driverId']),
        safe(row['constructorId']),
        safe(row['number']),
        safe(row['position']),
        safe(row['q1']),
        safe(row['q2']),
        safe(row['q3'])
    )
    for _, row in df_qualifying.iterrows()
]

cursor.executemany("""
    INSERT INTO qualifyings (id, race_id, driver_id, constructor_id, number, position, q1, q2, q3)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
""", qualifying_values)


In [12]:
pit_stop_values = [
    (
        safe(row['raceId']),
        safe(row['driverId']),
        safe(row['stop']),
        safe(row['lap']),
        safe(row['time']),
        safe(row['duration']),
        safe(row['milliseconds'])
    )
    for _, row in df_pit_stops.iterrows()
]

cursor.executemany("""
    INSERT INTO pit_stops (race_id, driver_id, stop, lap, time, duration, milliseconds)
    VALUES (%s, %s, %s, %s, %s, %s, %s)
""", pit_stop_values)


In [13]:
status_values = [
    (
        safe(row['statusId']),
        safe(row['status'])
    )
    for _, row in df_status.iterrows()
]

cursor.executemany("""
    INSERT INTO status (id, status)
    VALUES (%s, %s)
""", status_values)


In [14]:
results_values = [
    (
        safe(row['resultId']),
        safe(row['raceId']),
        safe(row['driverId']),
        safe(row['constructorId']),
        safe(row['number']),
        safe(row['grid']),
        safe(row['position']),
        safe(row['positionText']),
        safe(row['positionOrder']),
        safe(row['points']),
        safe(row['laps']),
        safe(row['time']),
        safe(row['milliseconds']),
        safe(row['fastestLap']),
        safe(row['rank']),
        safe(row['fastestLapTime']),
        safe(row['fastestLapSpeed']),
        safe(row['statusId'])
    )
    for _, row in df_results.iterrows()
]

cursor.executemany("""
    INSERT INTO results (
        id, race_id, driver_id, constructor_id, number, grid, position, position_text,
        position_order, points, laps, time, milliseconds, fastest_lap, rank,
        fastest_lap_time, fastest_lap_speed, status_id
    )
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
""", results_values)


In [15]:
lap_time_values = [
    (
        safe(row['raceId']),
        safe(row['driverId']),
        safe(row['lap']),
        safe(row['position']),
        safe(row['time']),
        safe(row['milliseconds'])
    )
    for _, row in df_lap_times.iterrows()
]

cursor.executemany("""
    INSERT INTO lap_times (race_id, driver_id, lap, position, time, milliseconds)
    VALUES (%s, %s, %s, %s, %s, %s)
""", lap_time_values)


In [16]:
driver_standings_values = [
    (
        safe(row['driverStandingsId']),
        safe(row['raceId']),
        safe(row['driverId']),
        safe(row['points']),
        safe(row['position']),
        safe(row['positionText']),
        safe(row['wins'])
    )
    for _, row in df_driver_standings.iterrows()
]

cursor.executemany("""
    INSERT INTO driver_standings (id, race_id, driver_id, points, position, position_text, wins)
    VALUES (%s, %s, %s, %s, %s, %s, %s)
""", driver_standings_values)


In [17]:
sprint_results_values = [
    (
        safe(row['resultId']),
        safe(row['raceId']),
        safe(row['driverId']),
        safe(row['constructorId']),
        safe(row['number']),
        safe(row['grid']),
        safe(row['position'])
    )
    for _, row in df_sprint_results.iterrows()
]

cursor.executemany("""
    INSERT INTO sprint_results (result_id, race_id, driver_id, constructor_id, number, grid, position)
    VALUES (%s, %s, %s, %s, %s, %s, %s)
""", sprint_results_values)


In [18]:
constructor_standings_values = [
    (
        safe(row['constructorStandingsId']),
        safe(row['raceId']),
        safe(row['constructorId']),
        safe(row['points']),
        safe(row['position']),
        safe(row['positionText']),
        safe(row['wins'])
    )
    for _, row in df_constructor_standings.iterrows()
]

cursor.executemany("""
    INSERT INTO constructor_standings (id, race_id, constructor_id, points, position, position_text, wins)
    VALUES (%s, %s, %s, %s, %s, %s, %s)
""", constructor_standings_values)


In [19]:
constructor_results_values = [
    (
        safe(row['constructorResultsId']),
        safe(row['raceId']),
        safe(row['constructorId']),
        safe(row['points']),
        safe(row['status'])
    )
    for _, row in df_constructor_results.iterrows()
]

cursor.executemany("""
    INSERT INTO constructor_results (id, race_id, constructor_id, points, status)
    VALUES (%s, %s, %s, %s, %s)
""", constructor_results_values)


# Verbindung zur Datenbank schließen
Durch `connection.commit()` werden alle zuvor ausgeführten Änderungen dauerhaft in der Datenbank gespeichert. Ohne diesen Befehl blieben die Einfügungen lediglich temporär. Anschließend wird die Verbindung geschlossen.

In [20]:
connection.commit()
cursor.close()
connection.close()